# NexusCommerce — Base Model Training

Trains a **Random Forest** demand-forecasting model on the **Store Item Demand Forecasting** dataset.

### Dataset required
Add this Kaggle competition dataset before running:
`https://www.kaggle.com/competitions/demand-forecasting-kernels-only`

### Output
All trained models are saved to `/kaggle/working/models/base/`.
Download them and copy to `nexuscommerce/ml/models/base/` in your project.

### Files produced
| File | Model |
|------|-------|
| `rf_model.pkl` | Random Forest |

In [ ]:
# No extra installs needed — scikit-learn is pre-installed on Kaggle

In [ ]:
import os
import pickle
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("All imports OK")

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────
OUTPUT_DIR   = "/kaggle/working/models/base"
DATASET_PATH = "/kaggle/input/demand-forecasting-kernels-only/train.csv"

FEATURE_COLS = ["month", "day", "day_of_week", "product_id",
                "lag_7", "lag_14", "rolling_mean_7"]
TARGET_COL   = "quantity"

TRAIN_RATIO      = 0.80
RF_N_ESTIMATORS  = 200
RANDOM_SEED      = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

---
## 1 — Load & Explore Dataset

In [ ]:
raw = pd.read_csv(DATASET_PATH)
print(f'Raw shape  : {raw.shape}')
print(f'Columns    : {list(raw.columns)}')
print(f'Date range : {raw["date"].min()}  →  {raw["date"].max()}')
print(f'Stores     : {raw["store"].nunique()}')
print(f'Items      : {raw["item"].nunique()}')
raw.head()

In [ ]:
# Quick EDA — total daily sales across all stores and items
raw['date'] = pd.to_datetime(raw['date'])
eda = raw.groupby('date')['sales'].sum()

fig, axes = plt.subplots(2, 1, figsize=(14, 6))

axes[0].plot(eda.index, eda.values, linewidth=0.8, color='steelblue')
axes[0].set_title('Total Daily Sales (all stores × items)', fontsize=13)
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
axes[0].set_ylabel('Units sold')
axes[0].grid(True, alpha=0.3)

monthly = eda.resample('M').sum()
axes[1].bar(monthly.index, monthly.values, width=20, color='coral', alpha=0.8)
axes[1].set_title('Monthly Total Sales', fontsize=13)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[1].set_ylabel('Units sold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/kaggle/working/eda_sales.png', dpi=120)
plt.show()
print('EDA chart saved.')

---
## 2 — Preprocessing

Aggregate to a **single daily time series** (total demand across all stores and items).  
This is the general demand pattern the base models will learn.

In [ ]:
# Normalise column names to NexusCommerce standard format
df = raw.rename(columns={'date': 'sale_date', 'sales': 'quantity'})
df['sale_date'] = pd.to_datetime(df['sale_date'])

# Daily aggregate
daily = (
    df.groupby('sale_date')['quantity']
    .sum()
    .reset_index()
    .set_index('sale_date')
    .asfreq('D')
    .ffill()
    .fillna(0)
)
daily.columns = ['quantity']

print(f'Daily rows : {len(daily)}')
print(f'Date range : {daily.index.min().date()}  →  {daily.index.max().date()}')
print(f'\nQuantity statistics:')
print(daily['quantity'].describe().round(2))

---
## 3 — Feature Engineering

Build the 7-feature matrix used by Random Forest:

| Feature | Description |
|---------|-------------|
| `month` | Calendar month (1-12) |
| `day` | Day of month (1-31) |
| `day_of_week` | Day of week (0=Mon ... 6=Sun) |
| `product_id` | Aggregated series uses 0 |
| `lag_7` | Sales 7 days ago |
| `lag_14` | Sales 14 days ago |
| `rolling_mean_7` | 7-day rolling average |

In [ ]:
def build_features(daily_df: pd.DataFrame) -> pd.DataFrame:
    f = daily_df.copy()
    f['month']          = f.index.month
    f['day']            = f.index.day
    f['day_of_week']    = f.index.dayofweek
    f['product_id']     = 0
    f['lag_7']          = f['quantity'].shift(7)
    f['lag_14']         = f['quantity'].shift(14)
    f['rolling_mean_7'] = f['quantity'].rolling(7).mean()
    return f.dropna()

features = build_features(daily)
print(f'Feature matrix : {features.shape}')
print(f'Columns        : {list(features.columns)}')
features.head(3)

---
## 4 — Train / Test Split

In [ ]:
split_idx  = int(len(features) * TRAIN_RATIO)
train      = features.iloc[:split_idx]
test       = features.iloc[split_idx:]

X_train    = train[FEATURE_COLS].values
y_train    = train[TARGET_COL].values
X_test     = test[FEATURE_COLS].values
y_test     = test[TARGET_COL].values

print(f"RF  train : {len(train):,} rows")
print(f"RF  test  : {len(test):,} rows")

In [ ]:
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, name: str) -> dict:
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mask = y_true != 0
    mape = float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100) \
           if mask.sum() > 0 else 0.0
    sep  = '=' * 48
    print(f'\n{sep}')
    print(f'  {name}')
    print(sep)
    print(f'  MAE   : {mae:>10.4f}')
    print(f'  RMSE  : {rmse:>10.4f}')
    print(f'  R²    : {r2:>10.4f}')
    print(f'  MAPE  : {mape:>9.2f} %')
    return {'model': name, 'mae': mae, 'rmse': rmse, 'r2_score': r2, 'mape': mape}


def plot_forecast(dates, y_true, y_pred, title, filename):
    plt.figure(figsize=(14, 4))
    plt.plot(dates, y_true, label='Actual',    color='steelblue', linewidth=1.2)
    plt.plot(dates, y_pred, label='Predicted', color='tomato',    linewidth=1.2, linestyle='--')
    plt.title(title, fontsize=13)
    plt.xlabel('Date'); plt.ylabel('Quantity')
    plt.legend(); plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'/kaggle/working/{filename}', dpi=120)
    plt.show()


all_metrics = []

---
## 5 — Random Forest Training

Random Forest uses the 7-feature matrix.
`warm_start=False` here — base model trains on the full general dataset from scratch.

In [ ]:
print('Training Random Forest...')

rf_model = RandomForestRegressor(
    n_estimators      = RF_N_ESTIMATORS,
    max_depth         = 15,
    min_samples_split = 5,
    min_samples_leaf  = 2,
    random_state      = RANDOM_SEED,
    n_jobs            = -1,
)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

rf_metrics = compute_metrics(y_test, rf_preds, 'Random Forest')
all_metrics.append(rf_metrics)

# Feature importance chart
imp_df = (
    pd.DataFrame({'feature': FEATURE_COLS, 'importance': rf_model.feature_importances_})
    .sort_values('importance', ascending=True)
)
plt.figure(figsize=(8, 4))
plt.barh(imp_df['feature'], imp_df['importance'], color='steelblue', alpha=0.8)
plt.xlabel('Importance'); plt.title('Random Forest — Feature Importance')
plt.tight_layout()
plt.savefig('/kaggle/working/rf_feature_importance.png', dpi=120)
plt.show()

plot_forecast(
    test.index, y_test, rf_preds,
    'Random Forest — Actual vs Predicted', 'rf_forecast.png'
)

rf_path = os.path.join(OUTPUT_DIR, 'rf_model.pkl')
with open(rf_path, 'wb') as f:
    pickle.dump(rf_model, f)
print(f'\n✅ Random Forest saved → {rf_path}')

---
## 6 — Model Metrics

In [ ]:
cmp_df = (
    pd.DataFrame([rf_metrics])
    .set_index("model")
    .round(4)
)

print("
" + "=" * 60)
print("  MODEL METRICS — NexusCommerce Base Models")
print("=" * 60)
print(cmp_df.to_string())

cmp_df.to_csv("/kaggle/working/model_metrics.csv")
print("
Metrics saved → /kaggle/working/model_metrics.csv")

In [ ]:
# Metrics bar chart
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ["steelblue", "coral", "mediumseagreen"]
metrics_to_plot = ["mae", "rmse", "mape"]

for ax, metric, color in zip(axes, metrics_to_plot, colors):
    val = cmp_df[metric].values[0]
    ax.bar([cmp_df.index[0]], [val], color=color, alpha=0.85)
    ax.set_title(metric.upper(), fontsize=12)
    ax.set_ylabel("Value")
    ax.text(0, val * 1.02, f"{val:.2f}", ha="center", va="bottom", fontsize=10)
    ax.grid(True, alpha=0.3, axis="y")

plt.suptitle("Random Forest — Metrics", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("/kaggle/working/rf_metrics_chart.png", dpi=120, bbox_inches="tight")
plt.show()

---
## 7 — Verify & List Saved Files

In [ ]:
print(f'
Files saved to  {OUTPUT_DIR}
')
print(f'{"File":<25}  {"Size"}')
print('-' * 40)
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    size  = os.path.getsize(fpath)
    label = f'{size / 1024:.1f} KB' if size < 1_000_000 else f'{size / 1_048_576:.1f} MB'
    print(f'{fname:<25}  {label}')

print('
' + '=' * 55)
print('  NEXT STEPS')
print('=' * 55)
print("""
1. Click "Output" (top-right of this notebook) and
   download the model file:
     rf_model.pkl

2. Copy it into your project at:
     nexuscommerce/ml/models/base/

3. Start the ML server:
     cd nexuscommerce/ml
     uvicorn main:app --reload
""")